## Removing an Atom Type

#### Example:
#### remove_atoms_from_cif('input.cif', 'output.cif', 'atom_type', amount)

In [3]:
import random

def remove_atoms_from_vasp(input_vasp, output_vasp, atom_type, amount_to_remove):
    with open(input_vasp, 'r') as f:
        lines = f.readlines()

    # Find atom types and counts
    atom_types = lines[5].split()
    atom_counts = [int(x) for x in lines[6].split()]
    if atom_type not in atom_types:
        raise ValueError(f"Atom type {atom_type} not found in VASP file.")

    atom_index = atom_types.index(atom_type)
    start = 8  # POSCAR format: atom coordinates start at line 9 (index 8)
    num_atoms = sum(atom_counts)
    atom_lines = lines[start:start+num_atoms]

    # Find indices of the specified atom type
    indices = []
    count = 0
    for i, t in enumerate(atom_types):
        for j in range(atom_counts[i]):
            if t == atom_type:
                indices.append(count)
            count += 1

    if len(indices) < amount_to_remove:
        raise ValueError(f"Not enough atoms of type {atom_type} to remove.")

    remove_indices = set(random.sample(indices, amount_to_remove))
    new_atom_lines = [line for i, line in enumerate(atom_lines) if i not in remove_indices]

    # Update atom count
    atom_counts[atom_index] -= amount_to_remove
    lines[6] = ' '.join(str(x) for x in atom_counts) + '\n'

    # Write new file
    new_lines = lines[:start] + new_atom_lines + lines[start+num_atoms:]
    with open(output_vasp, 'w') as f:
        f.writelines(new_lines)


In [18]:
remove_atoms_from_vasp('/Users/duygusenturk/Desktop/TU_Wien/SUPERCELL/2x2x2_supercell.vasp', '/Users/duygusenturk/Desktop/TU_Wien/SUPERCELL/100ps@300K/Ti-vacancies/NPT/10/10TiB2.vasp', 'Ti', 192)

## Generating Fluctuations

#### Example usage:
#### dislocate_atoms_in_vasp('input.vasp', 'output.vasp', 0, 0.02, unit='angstrom')

In [4]:
import numpy as np

def dislocate_atoms_in_vasp(input_vasp, output_vasp, min_dislocation, max_dislocation, unit='angstrom'):

    with open(input_vasp, 'r') as f:
        lines = f.readlines()

    # Find coordinate section
    for i, line in enumerate(lines):
        if line.strip().lower().startswith('direct') or line.strip().lower().startswith('cart'):
            coord_type = line.strip().lower()
            coord_start = i + 1
            break
    else:
        raise ValueError("Could not find coordinate section in VASP file.")

    # Get number of atoms
    atom_counts = [int(x) for x in lines[6].split()]
    num_atoms = sum(atom_counts)

    # Get lattice vectors
    scale = float(lines[1].strip())
    lattice = np.array([list(map(float, lines[j].split())) for j in range(2, 5)]) * scale

    # Dislocate atom positions
    new_atom_lines = []
    for j in range(num_atoms):
        tokens = lines[coord_start + j].split()
        pos = np.array(list(map(float, tokens[:3])))
        if coord_type.startswith('direct'):
            # Convert to Cartesian
            pos_cart = pos @ lattice
        else:
            pos_cart = pos

        # Generate random dislocation vector
        d = np.random.uniform(min_dislocation, max_dislocation)
        theta = np.random.uniform(0, 2*np.pi)
        phi = np.random.uniform(0, np.pi)
        dx = d * np.sin(phi) * np.cos(theta)
        dy = d * np.sin(phi) * np.sin(theta)
        dz = d * np.cos(phi)
        pos_cart += np.array([dx, dy, dz])

        # Convert back if needed
        if coord_type.startswith('direct'):
            pos_new = np.linalg.solve(lattice.T, pos_cart)
        else:
            pos_new = pos_cart

        # Keep only first 3 columns, preserve any selective dynamics flags
        new_line = "{:>12.8f} {:>12.8f} {:>12.8f}".format(*pos_new)
        if len(tokens) > 3:
            new_line += " " + " ".join(tokens[3:])
        new_atom_lines.append(new_line + "\n")

    # Write output
    new_lines = lines[:coord_start] + new_atom_lines + lines[coord_start + num_atoms:]
    with open(output_vasp, 'w') as f:
        f.writelines(new_lines)

    print(f"All atom positions have been randomly dislocated by {min_dislocation}-{max_dislocation} {unit}.")



In [23]:
dislocate_atoms_in_vasp('/Users/duygusenturk/Desktop/TU_Wien/supercell_validation/POSCARs/80', '/Users/duygusenturk/Desktop/TU_Wien/supercell_validation/B-interstitial/80', 0, 0.2, unit='angstrom')

All atom positions have been randomly dislocated by 0-0.2 angstrom.


## Replace Atom Ti => B
#### Example : replace_atom_type_in_vasp(input_vasp, output_vasp, old_atom_type, new_atom_type, amount_to_replace)

In [5]:
import random

def replace_atom_type_in_vasp(input_vasp, output_vasp, old_atom_type, new_atom_type, amount_to_replace):
   
    with open(input_vasp, 'r') as f:
        lines = f.readlines()

    atom_types = lines[5].split()
    atom_counts = [int(x) for x in lines[6].split()]
    
    # Check if old_atom_type exists in the file
    if old_atom_type not in atom_types:
        raise ValueError(f"Atom type {old_atom_type} not found in VASP file.")

    old_index = atom_types.index(old_atom_type)
    start = 8
    num_atoms = sum(atom_counts)
    atom_lines = lines[start:start+num_atoms]

    # Find indices of the specified atom type
    indices = []
    count = 0
    for i, t in enumerate(atom_types):
        for j in range(atom_counts[i]):
            if t == old_atom_type:
                indices.append(count)
            count += 1

    if len(indices) < amount_to_replace:
        raise ValueError(f"Not enough atoms of type {old_atom_type} to replace.")

    replace_indices = set(random.sample(indices, amount_to_replace))

    # Prepare new atom type order
    new_atom_types = atom_types.copy()
    new_atom_counts = atom_counts.copy()
    if new_atom_type in atom_types:
        new_index = atom_types.index(new_atom_type)
        new_atom_counts[new_index] += amount_to_replace
    else:
        new_atom_types.append(new_atom_type)
        new_atom_counts.append(amount_to_replace)
    new_atom_counts[old_index] -= amount_to_replace

    # Build new atom order
    atom_type_order = []
    count = 0
    for i, t in enumerate(atom_types):
        for j in range(atom_counts[i]):
            if t == old_atom_type and count in replace_indices:
                atom_type_order.append(new_atom_type)
            else:
                atom_type_order.append(t)
            count += 1

    # Group atom lines by type according to new order
    grouped_lines = {t: [] for t in new_atom_types}
    for t, line in zip(atom_type_order, atom_lines):
        grouped_lines[t].append(line)

    # Concatenate lines in the order of new_atom_types and new_atom_counts
    new_atom_lines = []
    for t, n in zip(new_atom_types, new_atom_counts):
        new_atom_lines.extend(grouped_lines[t][:n])

    # Update header
    lines[5] = ' '.join(new_atom_types) + '\n'
    lines[6] = ' '.join(str(x) for x in new_atom_counts) + '\n'

    # Write output
    new_lines = lines[:start] + new_atom_lines + lines[start+num_atoms:]
    with open(output_vasp, 'w') as f:
        f.writelines(new_lines)


In [16]:
replace_atom_type_in_vasp('/Users/duygusenturk/Desktop/TU_Wien/SUPERCELL/2x2x2_supercell.vasp', '/Users/duygusenturk/Desktop/TU_Wien/SUPERCELL/100ps@300K/B-interstitial/NPT/10/10TiB2.vasp', "Ti", "B", 192)

## Removing on a scaled range for interface investigation

In [1]:
def remove_atoms_in_x_region_vasp(input_vasp, output_vasp, atom_type, percentage_to_remove, forbidden_x_percent):
    """
    Removes a percentage of atoms of a given type from a VASP file,
    only if their x coordinate is NOT in the forbidden region (starting from the lowest x).
    forbidden_x_percent: float between 0 and 1, percentage of x axis that is forbidden for removal.
    percentage_to_remove: float between 0 and 1, fraction of allowed atoms to remove.
    """
    with open(input_vasp, 'r') as f:
        lines = f.readlines()

    atom_types = lines[5].split()
    atom_counts = [int(x) for x in lines[6].split()]
    if atom_type not in atom_types:
        raise ValueError(f"Atom type {atom_type} not found in VASP file.")

    atom_index = atom_types.index(atom_type)
    start = 8
    num_atoms = sum(atom_counts)
    atom_lines = lines[start:start+num_atoms]

    scale = float(lines[1].strip())
    lattice = [list(map(float, lines[j].split())) for j in range(2, 5)]
    lattice = [scale * np.array(vec) for vec in lattice]
    x_length = np.linalg.norm(lattice[0])

    coord_type = lines[7].strip().lower()
    is_direct = coord_type.startswith('direct')

    indices = []
    x_positions = []
    count = 0
    for i, t in enumerate(atom_types):
        for j in range(atom_counts[i]):
            if t == atom_type:
                tokens = atom_lines[count].split()
                pos = np.array(list(map(float, tokens[:3])))
                if is_direct:
                    pos_cart = pos @ np.array(lattice)
                else:
                    pos_cart = pos
                x_positions.append(pos_cart[0])
                indices.append(count)
            count += 1

    x_min = min(x_positions)
    x_max = max(x_positions)
    forbidden_width = forbidden_x_percent * (x_max - x_min)
    forbidden_x_limit = x_min + forbidden_width

    allowed_indices = [idx for idx, x in zip(indices, x_positions) if x > forbidden_x_limit]
    num_allowed = len(allowed_indices)
    amount_to_remove = int(round(percentage_to_remove * num_allowed))

    if num_allowed < amount_to_remove:
        raise ValueError(f"Not enough atoms of type {atom_type} outside forbidden region to remove.")

    remove_indices = set(random.sample(allowed_indices, amount_to_remove))
    new_atom_lines = [line for i, line in enumerate(atom_lines) if i not in remove_indices]

    atom_counts[atom_index] -= amount_to_remove
    lines[6] = ' '.join(str(x) for x in atom_counts) + '\n'

    new_lines = lines[:start] + new_atom_lines + lines[start+num_atoms:]
    with open(output_vasp, 'w') as f:
        f.writelines(new_lines)

    print(f"Removed {amount_to_remove} '{atom_type}' atoms ({percentage_to_remove*100:.1f}%) outside forbidden x region (first {forbidden_x_percent*100:.1f}% of x axis).")


In [10]:
remove_atoms_in_x_region_vasp("/Users/duygusenturk/Desktop/TU_Wien/DATA-LAMMPS/2x2x2_supercell.vasp", "/Users/duygusenturk/Desktop/TU_Wien/DATA-LAMMPS/interface@2500K/NPT /100/POSCAR.vasp", 
                              "Ti", 1, 0.3)

Removed 1344 'Ti' atoms (100.0%) outside forbidden x region (first 30.0% of x axis).
